In [ ]:
# ==============================================================================
# STEP 1: Install Required Libraries
# We drop deprecated packages ('sklearn') and keep only essential RAG libraries.
# ==============================================================================
!pip install -q datasets sentence-transformers rank_bm25

import json
import csv
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print("🚀 Environment ready. Hybrid Search components successfully installed.")

# ==============================================================================
# STEP 2: Swap PubMedQA for a Consumer Health Dataset (MedQuAD / WikiDoc)
# Pulling real consumer patient information that matches consumer query tones.
# ==============================================================================
print("📦 Downloading 'medalpaca/medical_meadow_wikidoc_patient_information' from Hugging Face...")
consumer_dataset = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information", split="train")

MASTER_CORPUS_FILE = 'corpus.jsonl'
EVALUATION_DATASET_FILE = 'dataset.csv'
TARGET_DISEASES = ["Asthma", "Diabetes", "Hypertension"]

# ==============================================================================
# OPTIMIZED PIPELINE: Sequential IDs + Overlap Resolution
# ==============================================================================
corpus_entries = []
evaluation_entries = []

# Clear, precise tracking counters to avoid index gaps
valid_doc_counter = 0
valid_query_counter = 0

def chunk_text_by_words(text, chunk_size=150, overlap=30):
    """Splits raw text strings into fixed-word windows with sliding boundaries."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        start += (chunk_size - overlap)
    return chunks

print("⚙️ Preprocessing text, applying sliding chunk windows, and resolving schemas...")
for idx, item in enumerate(consumer_dataset):
    real_patient_question = item.get('input', '')  # ✅ 'input' holds the real query
    output_text = item.get('output', '')

    # Skip empty entries safely
    if not real_patient_question.strip() or not output_text.strip():
        continue

    combined_text = f"{real_patient_question} {output_text}".lower()

    # Track all matched conditions instead of short-circuiting on the first one
    matched_conditions = [d for d in TARGET_DISEASES if d.lower() in combined_text]

    # Filter out entries with no target disease matches
    if not matched_conditions:
        continue

    # Standardize multi-topic documents (e.g., "Diabetes_Hypertension")
    # to avoid falsely blowing up one specific partition size
    detected_disease = "_".join(matched_conditions)

    # Process the text through the 150-word chunk splitting function
    text_chunks = chunk_text_by_words(output_text, chunk_size=150, overlap=30)
    chunk_ids_for_this_doc = []

    for chunk_idx, chunk_text in enumerate(text_chunks):
        doc_id = f"doc_consumer_{valid_doc_counter}"
        chunk_ids_for_this_doc.append(doc_id)

        corpus_entries.append({
            "doc_id": doc_id,
            "article_id": f"art_consumer_{idx}_chunk_{chunk_idx}",
            "title": real_patient_question[:50] + "...",
            "section": f"Patient Leaflet Section {chunk_idx + 1}",
            "text": chunk_text,
            "disease": detected_disease,
            "source_url": "https://huggingface.co"
        })
        valid_doc_counter += 1

    # Map the row array fields using the verified query string
    split_assignment = "dev" if valid_query_counter % 2 == 0 else "test"
    evaluation_entries.append({
        "question_id": f"Q_eval_{valid_query_counter}",
        "question": real_patient_question,  # ✅ Map the real question text here
        "gold_answers": json.dumps([output_text]),
        "gold_doc_ids": json.dumps(chunk_ids_for_this_doc),
        "split": split_assignment
    })
    valid_query_counter += 1

with open(MASTER_CORPUS_FILE, 'w', encoding='utf-8') as f:
    for entry in corpus_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"✅ Generated master consumer health database: '{MASTER_CORPUS_FILE}' ({len(corpus_entries)} passages).")

# ==============================================================================
# STEP 3: Physically Separate the Corpus Files by Granular Disease Grouping
# ==============================================================================
disease_partitions = defaultdict(list)

with open(MASTER_CORPUS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            norm_disease = record.get("disease", "Unknown")
            disease_partitions[norm_disease].append(record)

print("\n📂 Segmenting isolated sub-files by condition category:")
for disease, passages in disease_partitions.items():
    partition_filename = f"corpus_{disease.lower()}.jsonl"
    with open(partition_filename, 'w', encoding='utf-8') as pf:
        for passage in passages:
            pf.write(json.dumps(passage, ensure_ascii=False) + '\n')
    print(f" -> File created: {partition_filename} | Count: {len(passages)} passages.")

# ==============================================================================
# CLEAN STEP 4: Write Real Consumer Data to File (Without Mock Test Cases)
# ==============================================================================
print(f"📝 Writing clean evaluation dataset matrix to '{EVALUATION_DATASET_FILE}'...")

with open(EVALUATION_DATASET_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=["question_id", "question", "gold_answers", "gold_doc_ids", "split"])
    writer.writeheader()
    # Write only the real, parsed Hugging Face evaluation entries
    writer.writerows(evaluation_entries)

print(f"✅ Re-generated clean evaluation target indexes tracking table: '{EVALUATION_DATASET_FILE}'")

# ==============================================================================
# OPTIMIZED STEP 5 & 6: Dense Retrieval + Cross-Encoder Reranking
# Self-contained parsing logic to resolve variable initialization NameErrors.
# ==============================================================================
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# 1. Safely re-initialize the text data matrices from your local file partition
asthma_texts = []
asthma_ids = []
corpus_file_path = "corpus_asthma.jsonl"
evaluation_file_path = "dataset.csv"

try:
    with open(corpus_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                asthma_texts.append(record["text"])
                asthma_ids.append(record["doc_id"])
    print(f"📖 Loaded {len(asthma_texts)} documents from local '{corpus_file_path}' index pipeline.")
except FileNotFoundError:
    print(f"❌ Error: '{corpus_file_path}' missing. Please ensure your partitioning script ran first.")

# Safely load the evaluation dataset dataframe right here
try:
    eval_df = pd.read_csv(evaluation_file_path)
    print(f"📋 Loaded evaluation matrix framework from '{evaluation_file_path}'")
except FileNotFoundError:
    print(f"❌ Error: '{evaluation_file_path}' missing. Please ensure your evaluation generation script ran first.")

# 2. Re-verify model architectures
print("🧠 Loading models for neural semantic recall mapping...")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 3. Generate dense vector embeddings for the real asthma corpus
print("⚡ Vectorizing real asthma corpus matrix tracking indexes...")
asthma_embeddings = bi_encoder.encode(asthma_texts, show_progress_bar=False)

# 4. Search and retrieval system execution layer
def optimized_semantic_search(query_text, cross_encoder_threshold=-2.0, top_k_candidates=10):
    """
    1. Uses Dense Bi-Encoder to find documents that mean the same thing
       (e.g., matching 'triggers' to 'gets worse with').
    2. Uses Cross-Encoder to verify if it directly answers the question.
    """
    # Step 1: Dense Vector Retrieval (High Recall for paraphrases)
    query_embedding = bi_encoder.encode([query_text])
    similarities = cosine_similarity(query_embedding, asthma_embeddings)

    # Grab top 10 semantic candidates from the similarity matrix flat array
    top_candidate_indices = np.argsort(similarities[0])[::-1][:top_k_candidates]

    # Step 2: Cross-Encoder Verification (High Precision filtering)
    pairs = [[query_text, asthma_texts[idx]] for idx in top_candidate_indices]
    rerank_scores = cross_encoder.predict(pairs)

    validated_hits = []
    for rank_idx, corpus_idx in enumerate(top_candidate_indices):
        if rerank_scores[rank_idx] > cross_encoder_threshold:
            validated_hits.append(asthma_ids[corpus_idx])

    return validated_hits

# ==============================================================================
# Execute Evaluation Run Using Custom Explicit Test Probes
# Verifies pipeline precision against Direct, Paraphrased, Multi-Hop, and Refusal scenarios.
# ==============================================================================
print("\n🕵️ Executing Optimized Production Pipeline over Real Partitioned Data Matrices:")

# Statically defined test criteria to inspect retrieval accuracy boundaries
live_test_scenarios = [
    {
        "question_id": "TEST_DIRECT_001",
        "type": "Direct Question",
        "query": "What are common asthma triggers?"
    },
    {
        "question_id": "TEST_PARAPHRASE_002",
        "type": "Paraphrased Question",
        "query": "What can set off asthma symptoms?"
    },
    {
        "question_id": "TEST_MULTIHOP_003",
        "type": "Multi-Hop Question",
        "query": "Can exercise trigger asthma symptoms, and what does the source suggest about exercising with asthma?"
    },
    {
        "question_id": "TEST_INSUFFICIENT_004",
        "type": "Insufficient Evidence Check",
        "query": "Do any of the sources recommend a specific brand of air purifier for people with asthma?"
    }
]

for scenario in live_test_scenarios:
    q_id = scenario["question_id"]
    query = scenario["query"]
    q_type = scenario["type"]

    # Run the query through the Bi-Encoder + Cross-Encoder pipeline
    predicted_ids = optimized_semantic_search(query)

    print(f"\n[ID: {q_id}] ({q_type})")
    print(f" -> Query: '{query}'")
    print(f" -> Production Pipeline Retrieval: {predicted_ids}")

    # Clean verification logic based on content relevance metrics
    if q_id == "TEST_INSUFFICIENT_004":
        if not predicted_ids:
            print(" 🟢 SUCCESS: Insufficient evidence condition trapped cleanly. No hallucinations.")
        else:
            print(" ❌ FAIL: System leaked irrelevant documents for an unanswerable question.")

    elif q_id in ["TEST_DIRECT_001", "TEST_PARAPHRASE_002", "TEST_MULTIHOP_003"]:
        if len(predicted_ids) > 0:
            print(" 🟢 SUCCESS: Pipeline successfully bypassed vocab limits and surfaced the real document context!")

            # Safely look up text strings from the reconstructed asthma corpus arrays
            try:
                matched_idx = asthma_ids.index(predicted_ids[0])
                matched_text = asthma_texts[matched_idx]
                print(f" -> Top Match Text Snippet: \"{matched_text[:160]}...\"")
            except ValueError:
                print(" ⚠️ System traced matching Document ID, but document details were flushed from local memory vectors.")
        else:
            print(" ❌ FAIL: Pipeline completely missed the relevant medical passages.")
